# Sample Lists
- checking raw seqs 
- creating different sample lists that fit different conditions to use in assembly set
- purpose of this is to try assembling with similar sample types to get better results 

In [2]:
import numpy as np
import pandas as pd
import os
import re

In [3]:
os.chdir('/scratch/workspace/brooke_sienkiewicz_student_uml_edu-brooke-belseq/metadata')

In [4]:
ls

GW_Genohub_libraryprepped_submission_5799597.xlsx
GW_Genohub_submission_5799597.xlsx
Metagenomics_Tracker_Belize.csv


In [5]:
sample_data=pd.read_csv('Metagenomics_Tracker_Belize.csv', index_col=0)

In [6]:
sample_data.head()

,Health_Status,Starting_Weight,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_ng_ul,Microbe_Location,Microbe_clean_date/n,Host_ng_ul,Host_Location,...,Notes,Seq_date,Host_Seq_date,Microbe_seq_file,Host_seq_file,Seq_Location (in Unity),Sample_physical_location,Extraction_physical_location,Location_notes,Sample_Code = datecollected_tagnumber_transect_samplenumber_species
052022_BEL_CBC_T2_45_PSTR,Diseased_Margin,181,10_10_2023,22.8,10_17_2023,7.5,UML_NARWHAL_R2_B1,y,0.772,UML_NARWHAL_R6_B2,...,NaN,01_24; 10_19_23,NaN,052022_BEL_CBC_T2_45_PSTR,NaN,/project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_...,NaN,UML_NARWHAL_R2_B3,NaN,NaN
052022_BEL_CBC_T2_46_PSTR,Diseased_Tissue,162,10_10_2023,63.9,10_17_2023,18.5,UML_NARWHAL_R2_B1,y,too low,UML_NARWHAL_R6_B2,...,NaN,01_24; 10_19_23,NaN,052022_BEL_CBC_T2_46_PSTR,NaN,/project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_...,NaN,UML_NARWHAL_R2_B3,NaN,NaN
052022_BEL_CBC_T2_59_OFAV,Healthy,174,10_3_2023,73.7,10_16_2023,11.7,UML_NARWHAL_R2_B1,y,NaN,UML_NARWHAL_R6_B2,...,NaN,01_24; 10_19_23,NaN,052022_BEL_CBC_T2_59_OFAV,NaN,/project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_...,NaN,UML_NARWHAL_R2_B3,NaN,NaN
042024_BEL_CBC_T1_925_PAST,Healthy,325.5,8_19_2024,5.46,9_28_2024,1.57,UML_NARWHAL_R6_B30,9_28_2024,0.212,UML_NARWHAL_R6_B31,...,NaN,NaN,NaN,NaN,NaN,NaN,UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B29,AS - next to PAST 19,NaN
052022_BEL_CBC_T2_72_OFAV,Healthy,75,10_3_2023,47.6,10_11_23,3.96,UML_NARWHAL_R6_B1,10_12_23,0.522,UML_NARWHAL_R6_B2,...,NaN,01_24; 10_19_23,NaN,052022_BEL_CBC_T2_72_OFAV,NaN,/project/pi_sarah_gignouxwolfsohn_uml_edu/Raw_...,NaN,UML_NARWHAL_R2_B3,NaN,NaN


In [7]:
# make a column for year 

# Extract year from the sample names
def extract_year(sample_name):
    return sample_name.split('_')[0][2:]

# Create a new column 'Year' using the 'apply' function
sample_data['Year'] = sample_data.index.to_series().apply(extract_year)

# Print the updated DataFrame
print(sample_data['Year'])

052022_BEL_CBC_T2_45_PSTR      2022
052022_BEL_CBC_T2_46_PSTR      2022
052022_BEL_CBC_T2_59_OFAV      2022
042024_BEL_CBC_T1_925_PAST     2024
052022_BEL_CBC_T2_72_OFAV      2022
                               ... 
082024_BEL_CBC_T3_1561_MCAV    2024
082024_BEL_CBC_T3_1563_SSID    2024
082024_BEL_CBC_T3_1564_PAST    2024
052022_BEL_CBC_T3_16_SSID      2022
092023_BEL_CBC_T1_175_PAST     2023
Name: Year, Length: 524, dtype: object


In [8]:
# make column for species 
def extract_species(sample_name):
    return sample_name.split('_')[-1]  # Extract last element after splitting by "_"

sample_data['Species'] = sample_data.index.to_series().apply(extract_species)
print(sample_data['Species'])

052022_BEL_CBC_T2_45_PSTR      PSTR
052022_BEL_CBC_T2_46_PSTR      PSTR
052022_BEL_CBC_T2_59_OFAV      OFAV
042024_BEL_CBC_T1_925_PAST     PAST
052022_BEL_CBC_T2_72_OFAV      OFAV
                               ... 
082024_BEL_CBC_T3_1561_MCAV    MCAV
082024_BEL_CBC_T3_1563_SSID    SSID
082024_BEL_CBC_T3_1564_PAST    PAST
052022_BEL_CBC_T3_16_SSID      SSID
092023_BEL_CBC_T1_175_PAST     PAST
Name: Species, Length: 524, dtype: object


In [9]:
# remove unnecessary columns
columns_to_drop = ["Sample_Code = datecollected_tagnumber_transect_samplenumber_species",
                   'Date_Libprep','Host_Seq_date','Host_seq_file','Sample_physical_location','Extraction_physical_location','Location_notes']
sample_data = sample_data.drop(columns=columns_to_drop)
#print(sample_data.columns)

## samples sequenced in this run 
- make list of samples 
- make sure we have all seqs to match these samples 
- sequence date: nov 2024
- received seqs back: 03/17/2025 

In [10]:
# replace na with 'no'
sample_data['Seq_date']=sample_data['Seq_date'].fillna('no')
samples24=sample_data.loc[sample_data['Seq_date'].str.contains('24')]

In [11]:
# arrange by date and species
samples24=samples24.sort_values(by=['Year','Species'])

In [12]:
# export list of samples sequenced
samplelist24 = samples24.index.tolist()

In [13]:
# view lines 
samplelist24[20:25]

['062019_BEL_CBC_T3_16_MCAV',
 '062019_BEL_CBC_T3_6_MCAV',
 '062019_BEL_CBC_T3_8_MCAV',
 '062019_BEL_CBC_T3_9_MCAV',
 '062019_BEL_CBC_T1_10_MMEA']

### crosscheck seq files 

In [14]:
### crosscheck genohub submission form 
# samplelist24 - from metagenomics tracker
# genohublist24 - from genohub submission forms 

In [15]:
cd ..

/scratch/workspace/brooke_sienkiewicz_student_uml_edu-brooke-belseq


In [16]:
ls

03172025/          md5checksum             slurm-aws-32614617.out
aws                metadata/               slurm-checksum-30446035.out
genohublist24.txt  sample_list             slurm-checksum-32897719.out
md5_checksums.txt  slurm-aws-30186235.out


In [17]:
with open("genohublist24.txt", "r") as f:
    genohublist24 = [line.strip() for line in f]  # Removes extra spaces or newlines

print(genohublist24)
print(len(genohublist24)) 

['062019_BEL_CBC_T3_25_PAST', '122022_BEL_CBC_T1_133_PSTR', '122022_BEL_CBC_T2_116_PSTR', '122022_BEL_CBC_T4_35_PSTR', '122022_BEL_CBC_T2_99_PSTR', '122022_BEL_CBC_T1_123_OANN', '052022_BEL_CBC_T1_63_OFAV', '062019_BEL_CBC_T3_4_PAST', '062019_BEL_CBC_T1_10_MMEA', '062019_BEL_CBC_T1_14_MMEA', '062019_BEL_CBC_T2_12_MMEA', '062019_BEL_CBC_T2_13_MMEA', '052022_BEL_CBC_T1_39_MCAV', '052022_BEL_CBC_T1_54_MCAV', '052022_BEL_CBC_T1_62_MCAV', '052022_BEL_CBC_T2_12_MCAV', '052022_BEL_CBC_T2_4_MCAV', '052022_BEL_CBC_T2_56_MCAV', '122022_BEL_CBC_T1_151_MCAV', '122022_BEL_CBC_T2_86_MCAV', '122022_BEL_CBC_T2_88_MCAV', '122022_BEL_CBC_T2_92_MCAV', '122022_BEL_CBC_T2_95_MCAV', '122022_BEL_CBC_T3_119_MCAV', '122022_BEL_CBC_T3_128_MCAV', '122022_BEL_CBC_T3_142_MCAV', '122022_BEL_CBC_T3_155_MCAV', '122022_BEL_CBC_T4_3_MCAV', '062019_BEL_CBC_T2_14_MMEA', '062019_BEL_CBC_T2_15_MMEA', '122022_BEL_CBC_T4_1_OFAV', '122022_BEL_CBC_T3_133_MCAV', '122022_BEL_CBC_T4_14_MCAV', '122022_BEL_CBC_T4_5_MCAV', '052022_B

In [18]:
print(samplelist24)
len(samplelist24)

['062019_BEL_CBC_T1_16_MCAV', '062019_BEL_CBC_T1_17_MCAV', '062019_BEL_CBC_T1_20_MCAV', '062019_BEL_CBC_T1_22_MCAV', '062019_BEL_CBC_T1_24_MCAV', '062019_BEL_CBC_T1_3_MCAV', '062019_BEL_CBC_T1_4_MCAV', '062019_BEL_CBC_T1_6_MCAV', '062019_BEL_CBC_T1_9_MCAV', '062019_BEL_CBC_T2_16_MCAV', '062019_BEL_CBC_T2_18_MCAV', '062019_BEL_CBC_T2_23_MCAV', '062019_BEL_CBC_T2_28_MCAV', '062019_BEL_CBC_T2_5_MCAV', '062019_BEL_CBC_T2_8_MCAV', '062019_BEL_CBC_T2_9_MCAV', '062019_BEL_CBC_T3_1_MCAV', '062019_BEL_CBC_T3_11_MCAV', '062019_BEL_CBC_T3_14_MCAV', '062019_BEL_CBC_T3_15_MCAV', '062019_BEL_CBC_T3_16_MCAV', '062019_BEL_CBC_T3_6_MCAV', '062019_BEL_CBC_T3_8_MCAV', '062019_BEL_CBC_T3_9_MCAV', '062019_BEL_CBC_T1_10_MMEA', '062019_BEL_CBC_T1_14_MMEA', '062019_BEL_CBC_T2_12_MMEA', '062019_BEL_CBC_T2_13_MMEA', '062019_BEL_CBC_T2_14_MMEA', '062019_BEL_CBC_T2_15_MMEA', '062019_BEL_CBC_T2_6_MMEA', '062019_BEL_CBC_T2_7_MMEA', '062019_BEL_CBC_T3_20_MMEA', '062019_BEL_CBC_T3_22_MMEA', '062019_BEL_CBC_T3_3_MMEA'

220

In [19]:
# why do they have diff number of samples:
missing_samples = [sample for sample in genohublist24 if sample not in samplelist24]
print("Sample IDs that are NOT in metagenomics tracker:", missing_samples)

# just missing negatives that were sequenced 

Sample IDs that are NOT in metagenomics tracker: ['7_3_Neg', '7_11_Neg']


In [20]:
cd 03172025

/scratch/workspace/brooke_sienkiewicz_student_uml_edu-brooke-belseq/03172025


In [21]:
# num of seq files 
!ls -1 *R1_001.fastq.gz | wc -l

226


In [22]:
# 6 extra samples?

In [23]:
# save num & name of seq files 
!ls -1 *R1_001.fastq.gz > sample_list

In [24]:
mv sample_list ../sample_list

In [25]:
cd ..

/scratch/workspace/brooke_sienkiewicz_student_uml_edu-brooke-belseq


In [26]:
# upload sample_list 
with open("sample_list", "r") as f:
    sample_list = [line.strip() for line in f] 

In [27]:
# remove file endings and added sample number from sequencer 
# regex replacement
sample_list = [re.sub(r'_S.*_R.*_001.fastq.gz','', sample) for sample in sample_list]
print(sample_list)
# seqs in unity 

['052022_BEL_CBC_T1_10_PSTR', '052022_BEL_CBC_T1_11_PSTR', '052022_BEL_CBC_T1_12_MCAV', '052022_BEL_CBC_T1_13_MCAV', '052022_BEL_CBC_T1_1_PAST', '052022_BEL_CBC_T1_34_PAST', '052022_BEL_CBC_T1_35_OANN', '052022_BEL_CBC_T1_39_MCAV', '052022_BEL_CBC_T1_40_MCAV', '052022_BEL_CBC_T1_41_OANN', '052022_BEL_CBC_T1_4_PSTR', '052022_BEL_CBC_T1_52_PAST', '052022_BEL_CBC_T1_53_PAST', '052022_BEL_CBC_T1_54_MCAV', '052022_BEL_CBC_T1_55_PSTR', '052022_BEL_CBC_T1_57_MCAV', '052022_BEL_CBC_T1_60_MCAV', '052022_BEL_CBC_T1_61_PAST', '052022_BEL_CBC_T1_62_MCAV', '052022_BEL_CBC_T1_63_OFAV', '052022_BEL_CBC_T1_70_MCAV', '052022_BEL_CBC_T2_10_MCAV', '052022_BEL_CBC_T2_11_PAST', '052022_BEL_CBC_T2_12_MCAV', '052022_BEL_CBC_T2_13_PSTR', '052022_BEL_CBC_T2_14_PSTR', '052022_BEL_CBC_T2_45_PSTR', '052022_BEL_CBC_T2_46_PSTR', '052022_BEL_CBC_T2_4_MCAV', '052022_BEL_CBC_T2_56_MCAV', '052022_BEL_CBC_T2_59_OFAV', '052022_BEL_CBC_T2_5_PAST', '052022_BEL_CBC_T2_60_PAST', '052022_BEL_CBC_T2_62_PAST', '052022_BEL_CBC_T

In [28]:
# why do we have 4 extra samples in seq list 
missing_samples = [sample for sample in sample_list if sample not in genohublist24]
print("Sample IDs that are NOT in genohublist that are in seq files:", missing_samples)

Sample IDs that are NOT in genohublist that are in seq files: ['102019_BEL_CBC_T2_30_PSTR_host', '102019_BEL_CBC_T2_31_PSTR_host', 'Negative_extract_11-2-24', 'Undetermined']


In [29]:
missing_samples = [sample for sample in sample_list if sample not in samplelist24]
print("Sample IDs that are NOT in seq file that are in metagenomics tracker:", missing_samples)

Sample IDs that are NOT in seq file that are in metagenomics tracker: ['102019_BEL_CBC_T2_30_PSTR_host', '102019_BEL_CBC_T2_31_PSTR_host', '7_11_Neg', '7_3_Neg', 'Negative_extract_11-2-24', 'Undetermined']


In [30]:
missing_samples = [sample for sample in samplelist24 if sample not in sample_list]
print("Sample IDs that are NOT in metagenomics tracker that are in seq files:", missing_samples)

Sample IDs that are NOT in metagenomics tracker that are in seq files: []


In [31]:
# what is missing in our files 
missing_samples = [sample for sample in genohublist24 if sample not in sample_list]
print("Sequences missing:", missing_samples)

Sequences missing: []


In [32]:
# # remove leading 0 and see if they are still missing 
# missing_samples_nolead0 = [re.sub(r'^0(\d+)', r'\1', sample) for sample in missing_samples]
# print(missing_samples_nolead0)

In [33]:
# # what is missing in our files 
# missing_samples = [sample for sample in missing_samples_nolead0 if sample not in sample_list]
# print("Sequences missing:", missing_samples)

# # We have them all! (just some formatting/naming issues)

# ## jul 18, 2025 - have renamed all with leading 0 

### Make sample table

In [34]:
sample_list[0:5]

['052022_BEL_CBC_T1_10_PSTR',
 '052022_BEL_CBC_T1_11_PSTR',
 '052022_BEL_CBC_T1_12_MCAV',
 '052022_BEL_CBC_T1_13_MCAV',
 '052022_BEL_CBC_T1_1_PAST']

In [35]:
len(sample_list)

226

In [36]:
# match to sample metadata 

# load 
metadata=pd.read_csv('//project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/metadata/CBC_samples.csv')

#filter for UML samples (rna later or etoh) 
metadata=metadata[
    (metadata['Sample_type'] == 'Core_EtOH') |
    (metadata['Sample_type'] == 'Core_RNAlater')
]

#match list to tubelabel_species 
matched_metadata = metadata[metadata['Tubelabel_species'].isin(sample_list)]

In [37]:
matched_metadata.head()

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,Time_processed,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes
33,122022,BEL,CBC,12/4/22,CBC30N,1,NaN,22,OANN,NaN,NaN,Core_EtOH,120,Diseased_Margin,NaN,122022_BEL_CBC_T1_120_OANN,Depleted_UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B3,NaN,NaN
39,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,22,OANN,NaN,NaN,Core_EtOH,136,Diseased_Tissue,NaN,122022_BEL_CBC_T1_136_OANN,Depleted_UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B3,NaN,NaN
139,52022,BEL,CBC,5/21/22,CBC30N,1,22,22,OANN,NaN,NaN,Core_EtOH,41,Healthy,newly added May 2022,052022_BEL_CBC_T1_41_OANN,Depleted_UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B3,NaN,NaN
256,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,12,PSTR,NaN,NaN,Core_EtOH,122,Healthy,NaN,122022_BEL_CBC_T1_122_PSTR,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN
261,122022,BEL,CBC,12/4/22,CBC30N,1,NaN,6,PSTR,NaN,NaN,Core_EtOH,132,Diseased_Tissue,NaN,122022_BEL_CBC_T1_132_PSTR,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN


In [101]:
# add colony ID - t# newtagnum species
matched_metadata = matched_metadata.copy()
matched_metadata['TransectNum_str'] = 'T' + matched_metadata['TransectNum'].astype(str)
matched_metadata['colony_id'] = matched_metadata[['TransectNum_str', 'NewTagNum', 'Species']].astype(str).agg('_'.join, axis=1)
matched_metadata.drop(columns='TransectNum_str', inplace=True)

In [102]:
matched_metadata.columns

Index(['Month_year', 'Country', 'Location', 'CollectionDate', 'Transect',
       'TransectNum', 'OldTagNum', 'NewTagNum', 'Species', 'Time_sampled',
       'Time_processed', 'Sample_type', 'SampleNum', 'Health_status',
       'Sampling_notes', 'Tubelabel_species', 'Sample_physical_location',
       'Extraction_physical_location', 'Date_sequenced', 'Notes', 'colony_id'],
      dtype='object')

In [103]:
matched_metadata[['Month_year','CollectionDate','Transect','TransectNum','NewTagNum',
                  'Species','SampleNum','Health_status','Sample_type','Tubelabel_species']]

,Month_year,CollectionDate,Transect,TransectNum,NewTagNum,Species,SampleNum,Health_status,Sample_type,Tubelabel_species
33,122022,12/4/22,CBC30N,1,22,OANN,120,Diseased_Margin,Core_EtOH,122022_BEL_CBC_T1_120_OANN
39,122022,12/2/22,CBC30N,1,22,OANN,136,Diseased_Tissue,Core_EtOH,122022_BEL_CBC_T1_136_OANN
139,52022,5/21/22,CBC30N,1,22,OANN,41,Healthy,Core_EtOH,052022_BEL_CBC_T1_41_OANN
256,122022,12/2/22,CBC30N,1,12,PSTR,122,Healthy,Core_EtOH,122022_BEL_CBC_T1_122_PSTR
261,122022,12/4/22,CBC30N,1,6,PSTR,132,Diseased_Tissue,Core_EtOH,122022_BEL_CBC_T1_132_PSTR
...,...,...,...,...,...,...,...,...,...,...
1129,62019,6/21/19,SR30N,2,68,PAST,2,Healthy,Core_EtOH,062019_BEL_CBC_T2_2_PAST
1141,62019,6/21/19,SR30N,2,59,MCAV,5,Healthy,Core_EtOH,062019_BEL_CBC_T2_5_MCAV
1144,62019,6/21/19,SR30N,2,330,MMEA,6,Healthy,Core_EtOH,062019_BEL_CBC_T2_6_MMEA
1146,62019,6/21/19,SR30N,2,344,MMEA,7,Healthy,Core_EtOH,062019_BEL_CBC_T2_7_MMEA


In [104]:
# make sample table 
# unique species, transect, health statuses, timepoint 
    # leaving monthyear as is for now 


In [105]:
matched_metadata['Month_year'].unique()

[122022, 52022, 102019, 62019]
Categories (4, int64): [62019 < 102019 < 52022 < 122022]

In [106]:
# combine 062019 and 102019? 

In [107]:
month_order = [62019, 102019, 52022, 122022]
matched_metadata['Month_year'] = pd.Categorical(
    matched_metadata['Month_year'], categories=month_order, ordered=True
)

In [108]:
summary_table = (
    matched_metadata
    .groupby(['Month_year', 'Transect', 'Species', 'Health_status'])
    .size()
    .reset_index(name='n')
    .pivot_table(index=['Month_year', 'Transect', 'Health_status'],
                 columns='Species',
                 values='n',
                 fill_value=0)
    .astype(int)
)

summary_table.index = summary_table.index.set_names(['Month_year', 'Transect', 'Health_status'])
summary_table = summary_table.reset_index()

summary_table = summary_table.set_index('Month_year')
summary_table.columns.name = None  # removes 'Species' label above columns

# remove rows with all 0s
species_cols = ['MCAV', 'MMEA', 'OANN', 'OFAV', 'PAST', 'PSTR']
summary_table = summary_table.loc[~(summary_table[species_cols] == 0).all(axis=1)]

/tmp/ipykernel_3559972/3477100914.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['Month_year', 'Transect', 'Species', 'Health_status'])
/tmp/ipykernel_3559972/3477100914.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(index=['Month_year', 'Transect', 'Health_status'],


In [109]:
summary_table

,Transect,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
Month_year,,,,,,,,
62019,CBC30N,Healthy,9,2,0,0,5,0
62019,Lagoon,Healthy,8,5,0,0,9,0
62019,SR30N,Healthy,7,6,0,0,6,0
102019,CBC30N,Healthy,0,0,0,0,0,7
102019,Lagoon,Healthy,0,0,0,0,0,7
102019,SR30N,Healthy,0,0,0,0,0,7
52022,CBC30N,Diseased_Margin,3,0,0,0,0,1
52022,CBC30N,Diseased_Tissue,3,0,0,0,1,1
52022,CBC30N,Healthy,3,0,2,1,4,2


In [ ]:
# just pre- and post- disease for each sp 

In [124]:
# group 2019s and 2022s
table = summary_table.copy()
table['Year'] = table.index.astype(str).str[-4:].astype(int)

In [125]:
# sort by location
condensed2 = (
    table
    .groupby(['Year', 'Transect','Health_status']) 
    .sum(numeric_only=True)
    .reset_index()
)
condensed2

,Year,Transect,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
0,2019,CBC30N,Healthy,9,2,0,0,5,7
1,2019,Lagoon,Healthy,8,5,0,0,9,7
2,2019,SR30N,Healthy,7,6,0,0,6,7
3,2022,CBC30N,Diseased_Margin,3,0,1,0,1,2
4,2022,CBC30N,Diseased_Tissue,4,0,1,0,2,2
5,2022,CBC30N,Healthy,5,0,4,1,8,3
6,2022,CURLEW,Diseased_Margin,2,0,0,1,0,2
7,2022,CURLEW,Diseased_Tissue,2,0,0,1,0,2
8,2022,CURLEW,Healthy,3,0,0,3,0,3
9,2022,Lagoon,Diseased_Margin,4,0,0,0,2,0


In [126]:
# just pre and post per sp (no location)
condensed = (
    table
    .groupby(['Year', 'Health_status'])
    .sum(numeric_only=True)
    .reset_index()
)
condensed

,Year,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
0,2019,Healthy,24,13,0,0,20,21
1,2022,Diseased_Margin,9,0,2,2,4,6
2,2022,Diseased_Tissue,11,0,3,2,5,6
3,2022,Healthy,24,0,8,15,24,21


#### investigate colony numbers
- ex: why are there 24 healthy mcav samples in 2019 AND 2022? 

##### **MCAV**

In [111]:
# crosscheck 2019 and 2022 colonies and samples

# start with mcav subset 
mcav=matched_metadata[matched_metadata['Species']=="MCAV"]
# list of all samples in 2019
meta_2019=mcav[
    (mcav['Month_year']==62019) |
    (mcav['Month_year']==102019)]
ids_2019=set(meta_2019['colony_id'].unique())
# list of all samples in 2022
meta_2022=mcav[
    (mcav['Month_year']==52022) |
    (mcav['Month_year']==122022)]
ids_2022=set(meta_2022['colony_id'].unique())
# do they match 
ids_2019 == ids_2022

False

In [137]:
# total mcav colonies 
print(len(mcav['colony_id'].unique()))

# colonies present in both years
print(len(ids_2019 & ids_2022))
# 17 shared

print(len(ids_2019))
# 24 initial

print(len(ids_2022))
# 7 added, 7 died 

31
17
24
24


In [123]:
# colonies not in 2022 
print('colonies not in 2022:',len(ids_2019 - ids_2022), 'colonies',
      ids_2019 - ids_2022)
# manually checking fate 
# all died in 052022

# new colonies in 2022 not present in 2019
print('new colonies in 2022 not present in 2019:',len(ids_2022 - ids_2019), 'colonies',
      ids_2022 - ids_2019)
# manually checking fate 
# all tagged in 122022

colonies not in 2022: 7 colonies {'T1_342_MCAV', 'T1_333_MCAV', 'T1_329_MCAV', 'T1_355_MCAV', 'T2_56_MCAV', 'T3_9_MCAV', 'T3_12_MCAV'}
new colonies in 2022 not present in 2019: 7 colonies {'T4_28_MCAV', 'T4_30_MCAV', 'T3_71_MCAV', 'T4_95_MCAV', 'T4_94_MCAV', 'T3_67_MCAV', 'T4_76_MCAV'}


In [ ]:
# there just happen to be 7 mcav that died and 7 that were added

In [128]:
# check healthy 

# healthy 2019 samples 
healthy_mcav2019=meta_2019[meta_2019['Health_status']=='Healthy']
ids_h2019=set(healthy_mcav2019['colony_id'].unique())

# healthy 2022 samples 
healthy_mcav2022=meta_2022[meta_2022['Health_status']=='Healthy']
ids_h2022=set(healthy_mcav2022['colony_id'].unique())

# do they match 
ids_h2019 == ids_h2022

False

In [141]:
# colonies present in both years
print(len(ids_h2019 & ids_h2022))
# 11 stayed healthy 

print(len(ids_h2019))
# ALL started as healthy 

print(len(ids_h2022))

11
24
15
False


In [143]:
# healthy colonies not in 2022 
h2019disappeared=ids_h2019 - ids_h2022
print('healthy colonies that were not healthy in 2022:',len(ids_h2019 - ids_h2022), 'colonies',
      h2019disappeared)
# manually checking fate 
# died (5 colonies): 300s tags, t2_56, t3_9, t3_12
# got disease: t3_17, t1_15, t3_22, t1_8, t1_14, t3_15
# t1_8 seems to be the only one that has a disease sample from both 05 and 122022

# new healthy colonies in 2022 not present in 2019
print('new healthy colonies in 2022 not present in 2019:',len(ids_h2022 - ids_h2019), 'colonies',
      ids_h2022 - ids_h2019)
# manually checking fate 
# 4 of the 7 newly added were healthy in 2022 so this makes sense 

healthy colonies that were not healthy in 2022: 13 colonies {'T1_342_MCAV', 'T1_333_MCAV', 'T1_329_MCAV', 'T1_355_MCAV', 'T3_17_MCAV', 'T2_56_MCAV', 'T3_9_MCAV', 'T1_15_MCAV', 'T3_12_MCAV', 'T3_22_MCAV', 'T1_8_MCAV', 'T1_14_MCAV', 'T3_15_MCAV'}
new healthy colonies in 2022 not present in 2019: 4 colonies {'T4_30_MCAV', 'T4_76_MCAV', 'T3_71_MCAV', 'T4_28_MCAV'}


In [155]:
# weird ones: 
mcav[mcav['colony_id']=='T1_8_MCAV']

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
302,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,8,MCAV,NaN,...,Core_EtOH,144,Diseased_Tissue,NaN,122022_BEL_CBC_T1_144_MCAV,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN,T1_8_MCAV
641,52022,BEL,CBC,5/21/22,CBC30N,1,390,8,MCAV,NaN,...,Core_EtOH,12,Diseased_Margin,NaN,052022_BEL_CBC_T1_12_MCAV,Depleted_ UML_NARWHAL_R1_B3,DNA_extracted,sequenced,Only sand in tissue sample,T1_8_MCAV
643,52022,BEL,CBC,5/21/22,CBC30N,1,390,8,MCAV,NaN,...,Core_EtOH,13,Diseased_Tissue,NaN,052022_BEL_CBC_T1_13_MCAV,Depleted_UML_NARWHAL_R1_B3,DNA_extracted,sequenced,Only sand in tissue sample,T1_8_MCAV
929,62019,BEL,CBC,6/24/19,CBC30N,1,390,8,MCAV,NaN,...,Core_EtOH,16,Healthy,mucus sheaths,062019_BEL_CBC_T1_16_MCAV,UML_NARWHAL_R1_B1,DNA_extracted,sequenced,NaN,T1_8_MCAV


##### **PSTR**

In [157]:
# crosscheck 2019 and 2022 colonies and samples

# pstr subset 
pstr=matched_metadata[matched_metadata['Species']=="PSTR"]
# list of all samples in 2019
meta_2019=pstr[
    (pstr['Month_year']==62019) |
    (pstr['Month_year']==102019)]
ids_2019=set(meta_2019['colony_id'].unique())
# list of all samples in 2022
meta_2022=pstr[
    (pstr['Month_year']==52022) |
    (pstr['Month_year']==122022)]
ids_2022=set(meta_2022['colony_id'].unique())
# do they match 
ids_2019 == ids_2022

False

In [158]:
# total pstr colonies 
print(len(pstr['colony_id'].unique()))

# colonies present in both years
print(len(ids_2019 & ids_2022))
# 11 shared

print(len(ids_2019))
# 21 initial

print(len(ids_2022))
# 10 added, 10 died in 5/22

31
11
21
21


In [160]:
# confirming that there 10 added and 10 died - yes 

# colonies not in 2022 
print('colonies not in 2022:',len(ids_2019 - ids_2022), 'colonies',
      ids_2019 - ids_2022)
# manually checking fate 
# all died in 052022

# new colonies in 2022 not present in 2019
print('new colonies in 2022 not present in 2019:',len(ids_2022 - ids_2019), 'colonies',
      ids_2022 - ids_2019)
# manually checking fate 
# all tagged in 2022

colonies not in 2022: 10 colonies {'T1_417_PSTR', 'T3_30_PSTR', 'T1_419_PSTR', 'T1_422_PSTR', 'T1_404_PSTR', 'T3_16_PSTR', 'T2_403_PSTR', 'T3_32_PSTR', 'T2_427_PSTR', 'T3_11_PSTR'}
new colonies in 2022 not present in 2019: 10 colonies {'T3_70_PSTR', 'T4_79_PSTR', 'T2_28_PSTR', 'T4_96_PSTR', 'T4_98_PSTR', 'T2_32_PSTR', 'T4_97_PSTR', 'T3_74_PSTR', 'T4_80_PSTR', 'T3_75_PSTR'}


In [163]:
# check healthy 

# healthy 2019 samples 
healthy_pstr2019=meta_2019[meta_2019['Health_status']=='Healthy']
ids_h2019=set(healthy_pstr2019['colony_id'].unique())

# healthy 2022 samples 
healthy_pstr2022=meta_2022[meta_2022['Health_status']=='Healthy']
ids_h2022=set(healthy_pstr2022['colony_id'].unique())

# do they match 
ids_h2019 == ids_h2022

False

In [164]:
# colonies present in both years
print(len(ids_h2019 & ids_h2022))
# 9 stayed healthy 

print(len(ids_h2019))

print(len(ids_h2022))

9
21
16


##### **PAST**

In [198]:
# crosscheck 2019 and 2022 colonies and samples
# no past colonies were added in 2022 so why are there 24 healthy in 2022 and 20 healthy in 2019? 

# past subset 
past=matched_metadata[matched_metadata['Species']=="PAST"]
# list of all samples in 2019
meta_2019=past[
    (past['Month_year']==62019) |
    (past['Month_year']==102019)]
ids_2019=set(meta_2019['colony_id'].unique())
# list of all samples in 2022
meta_2022=past[
    (past['Month_year']==52022) |
    (past['Month_year']==122022)]
ids_2022=set(meta_2022['colony_id'].unique())
# do they match 
ids_2019 == ids_2022

False

In [200]:
# total past colonies 
print(len(past['colony_id'].unique()))

# colonies present in both years
print(len(ids_2019 & ids_2022))
# 16 shared

print(len(ids_2019))
# 20 initial

print(len(ids_2022))
# none added, lost 4
# 9 colonies had a sample in 52022 and 122022 
16+9 
#i'm confused 

20
16
20
16


25

In [201]:
# confirming - no colonies added, 2 died, and 2 weren't sampled again for some reason

# colonies not in 2022 
print('colonies not in 2022:',len(ids_2019 - ids_2022), 'colonies',
      ids_2019 - ids_2022)
# manually checking fate 
# 347 and 12flag died 
# see below about t2_47 and t3_13

# new colonies in 2022 not present in 2019
print('new colonies in 2022 not present in 2019:',len(ids_2022 - ids_2019), 'colonies',
      ids_2022 - ids_2019)
# none added in 2022

colonies not in 2022: 4 colonies {'T3_13_PAST', 'T2_347_PAST', 'T3_12flag_PAST', 'T2_47_PAST'}
new colonies in 2022 not present in 2019: 0 colonies set()


In [202]:
# weird ones: 
past[past['colony_id']=='T2_47_PAST']
# only a sample from 2019 but never has a mortality date 

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
915,62019,BEL,CBC,6/25/19,SR30N,2,47,47,PAST,NaN,...,Core_EtOH,29,Healthy,orange tag,062019_BEL_CBC_T2_29_PAST,UML_NARWHAL_R1_B1,DNA_extracted,NaN,NaN,T2_47_PAST


In [203]:
# weird ones: 
past[past['colony_id']=='T3_13_PAST']
# only a sample from 2019 but never has a mortality date 

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
967,62019,BEL,CBC,6/23/19,Lagoon,3,358,13,PAST,NaN,...,Core_EtOH,10,Healthy,NaN,062019_BEL_CBC_T3_10_PAST,UML_NARWHAL_R1_B1,penguin,NaN,NaN,T3_13_PAST


In [204]:
# weird ones: 
past[past['colony_id']=='T3_24_PAST']
# check that this healthy 12/2022 sample is correct? 
# colony data says this colony died in 5/22 but health statuses say healthy through 12/2022

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
431,122022,BEL,CBC,12/3/22,Lagoon,3,NaN,24,PAST,NaN,...,Core_EtOH,122,Healthy,NaN,122022_BEL_CBC_T3_122_PAST,UML_NARWHAL_R1_B5,UML_NARWHAL_R2_B29,NaN,NaN,T3_24_PAST
995,62019,BEL,CBC,6/23/19,Lagoon,3,314,24,PAST,NaN,...,Core_EtOH,23,Healthy,NaN,062019_BEL_CBC_T3_23_PAST,UML_NARWHAL_R1_B1,penguin,NaN,NaN,T3_24_PAST


In [205]:
# check healthy 

# healthy 2019 samples 
healthy_past2019=meta_2019[meta_2019['Health_status']=='Healthy']
ids_h2019=set(healthy_past2019['colony_id'].unique())

# healthy 2022 samples 
healthy_past2022=meta_2022[meta_2022['Health_status']=='Healthy']
ids_h2022=set(healthy_past2022['colony_id'].unique())

# do they match 
ids_h2019 == ids_h2022

False

In [206]:
# colonies present in both years
print(len(ids_h2019 & ids_h2022))
# 5 stayed healthy 

print(len(ids_h2019))

print(len(ids_h2022))

15
20
15


In [207]:
print('healthy colonies that were not healthy in 2022:',len(ids_h2019 - ids_h2022), 'colonies',
      ids_h2019 - ids_h2022)
print('only difference b/w above difference and this healthy list:', ids_2022-ids_h2022)
# manually checking fate 

# new healthy colonies in 2022 not present in 2019
print('new healthy colonies in 2022 not present in 2019:',len(ids_h2022 - ids_h2019), 'colonies',
      ids_h2022 - ids_h2019)

healthy colonies that were not healthy in 2022: 5 colonies {'T3_12flag_PAST', 'T3_13_PAST', 'T2_347_PAST', 'T3_18_PAST', 'T2_47_PAST'}
only difference b/w above difference and this healthy list: {'T3_18_PAST'}
new healthy colonies in 2022 not present in 2019: 0 colonies set()


In [208]:
# weird ones: 
past[past['colony_id']=='T3_18_PAST']
# diseased 

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
424,122022,BEL,CBC,12/3/22,Lagoon,3,NaN,18,PAST,NaN,...,Core_EtOH,115,Diseased_Tissue,NaN,122022_BEL_CBC_T3_115_PAST,UML_NARWHAL_R1_B5,UML_NARWHAL_R2_B26,NaN,NaN,T3_18_PAST
438,122022,BEL,CBC,12/3/22,Lagoon,3,NaN,18,PAST,NaN,...,Core_EtOH,129,Diseased_Margin,NaN,122022_BEL_CBC_T3_129_PAST,UML_NARWHAL_R1_B5,UML_NARWHAL_R2_B26,NaN,NaN,T3_18_PAST
786,52022,BEL,CBC,5/20/22,Lagoon,3,365,18,PAST,NaN,...,Core_EtOH,54,Diseased_Margin,"TL (F, SA)",052022_BEL_CBC_T3_54_PAST,UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B26,NaN,NaN,T3_18_PAST
787,52022,BEL,CBC,5/20/22,Lagoon,3,365,18,PAST,NaN,...,Core_EtOH,55,Diseased_Tissue,"TL (F, SA)",052022_BEL_CBC_T3_55_PAST,UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B26,NaN,NaN,T3_18_PAST
971,62019,BEL,CBC,6/23/19,Lagoon,3,365,18,PAST,NaN,...,Core_EtOH,12,Healthy,NaN,062019_BEL_CBC_T3_12_PAST,UML_NARWHAL_R1_B1,UML_NARWHAL_R2_B29,NaN,NaN,T3_18_PAST


In [209]:
# so only 1 2019 healthy colony got disease in 2022? 
# so why is there 4 more healthy samples in 2022 than in 2019??
past_meta=matched_metadata[matched_metadata['Species']=='PAST']
len(past_meta)
# 53 total samples 

53

In [210]:
past_metah=past_meta[past_meta['Health_status']=="Healthy"]
print(len(past_metah))
# 44 total healthy samples 
len(past_metah['colony_id'].unique())
# 20 unique colonies 
# do some have duplicate healthy samples in 2022??

44


20

In [214]:
# sep 5 and 122022 - are there any colony IDs that have samples from both dates? 
pasth_52022=healthy_past2022[(healthy_past2022['Month_year']==52022)]
pasth_122022=healthy_past2022[(healthy_past2022['Month_year']==122022)]

ids_52022=set(pasth_52022['colony_id'].unique())
ids_2022=set(pasth_122022['colony_id'].unique())

print('healthy colonies sampled in BOTH 5/22 and 12/22:',ids_52022 & ids_2022)

healthy colonies sampled in BOTH 5/22 and 12/22: {'T1_13_PAST', 'T3_10_PAST', 'T1_21_PAST', 'T3_8_PAST', 'T2_63_PAST', 'T1_2_PAST', 'T3_6_PAST', 'T2_57_PAST', 'T2_68_PAST'}


In [212]:
past[past['colony_id']=='T1_13_PAST']

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
310,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,13,PAST,NaN,...,Core_EtOH,152,Healthy,NaN,122022_BEL_CBC_T1_152_PAST,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B29,NaN,NaN,T1_13_PAST
713,52022,BEL,CBC,5/21/22,CBC30N,1,395,13,PAST,NaN,...,Core_EtOH,53,Healthy,NaN,052022_BEL_CBC_T1_53_PAST,UML_NARWHAL_R1_B3,penguin,NaN,NaN,T1_13_PAST
921,62019,BEL,CBC,6/24/19,CBC30N,1,395,13,PAST,NaN,...,Core_EtOH,12,Healthy,NaN,062019_BEL_CBC_T1_12_PAST,UML_NARWHAL_R1_B1,penguin,NaN,NaN,T1_13_PAST


In [213]:
past[past['colony_id']=='T1_20_PAST']

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
263,122022,BEL,CBC,12/4/22,CBC30N,1,NaN,20,PAST,NaN,...,Core_EtOH,134,Diseased_Margin,NaN,122022_BEL_CBC_T1_134_PAST,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN,T1_20_PAST
267,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,20,PAST,NaN,...,Core_EtOH,139,Diseased_Tissue,NaN,122022_BEL_CBC_T1_139_PAST,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN,T1_20_PAST
724,52022,BEL,CBC,5/21/22,CBC30N,1,386,20,PAST,NaN,...,Core_EtOH,61,Healthy,NaN,052022_BEL_CBC_T1_61_PAST,UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B29,NaN,NaN,T1_20_PAST
933,62019,BEL,CBC,6/24/19,CBC30N,1,386,20,PAST,NaN,...,Core_EtOH,18,Healthy,NaN,062019_BEL_CBC_T1_18_PAST,UML_NARWHAL_R1_B1,penguin,NaN,NaN,T1_20_PAST


### Match with colony data?

In [335]:
# upload and clean up colony data 
colony=pd.read_csv('//project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/metadata/CBC_ColonyData.csv')
colony = colony.iloc[:, 1:]

# add colony ID - t# newtagnum species
colony = colony.copy()
colony['TransectNum_str'] = 'T' + colony['TransectNum'].astype(str)
colony['colony_id'] = colony[['TransectNum_str', 'NewTagNum', 'Species']].astype(str).agg('_'.join, axis=1)
colony.drop(columns='TransectNum_str', inplace=True)

# remove bb and hangman
colony=colony[
    (colony['Transect']!='BB') &
    (colony['Transect']!='HANGMAN')]

#### PAST 

In [336]:
# filter for past 
past_colony=colony[colony['Species']=="PAST"]

In [ ]:
# merge dfs 
merged = pd.merge(
    conditions_long,
    past_samples,
    on=['colony_id', 'Month_year'],
    how='outer'
)
# original merge for below 

In [373]:
# get health status from each date (2019 - 2022) from colony data 
past_conditions=past_colony.loc[:,('colony_id','Date_InitialTag','Date_DocumentedDisease','Date_DocumentedMortality',
                   '062019_Condition','052022_Condition','122022_Condition')]

# pivot to match sample data setup
conditions_long = past_conditions.melt(
    id_vars='colony_id',
    value_vars=['062019_Condition', '052022_Condition', '122022_Condition'],
    var_name='Month_year',
    value_name='colony_condition'
)
# match month year format 
conditions_long['Month_year'] = conditions_long['Month_year'].str.extract(r'(\d+)').astype(int)

# get health status of samples from each date 
past_samples=past.loc[:,('colony_id','Month_year','Health_status','Sampling_notes','Tubelabel_species')]
# change col name 
past_samples['sample_condition']=past_samples['Health_status']

# merge dfs
merged = pd.merge(
    conditions_long.merge(past_conditions, on='colony_id', how='left'),
    past_samples,
    on=['colony_id', 'Month_year'],
    how='outer'
)

# need to create rules to match 
# if colony_condition is healthy sample = healthy 
# if condition is diseased healthy = diseased_margin and diseased_tissue (but rn these are on 2 diff rows)
# if condition is Not_visited, sample = NaN
# if condition is Dead, sample = NaN 
def status_match(group):
    # get the condition and list sample statuses of each colony at each monthyear (by row) 
    cond = group['colony_condition'].iloc[0]
    statuses = group['Health_status'].dropna().tolist()

    # match colony conditions to sample conditions 
    if cond == 'Healthy':
        return all(s == 'Healthy' for s in statuses)
    elif cond == 'Diseased':
        return all(x in statuses for x in ['Diseased_Tissue', 'Diseased_Margin'])
    elif cond in ['Dead', 'Not_Visited']:
        return all(pd.isna(s) for s in group['Health_status'])
        
    else:
        return False

In [379]:
# run function (status_match) to match conditions
rows = []
# check each 'group' (unique combos of colony and monthyear) and store results 
for (colony_id, month), group in merged.groupby(['colony_id', 'Month_year']):
    rows.append({
        'colony_id': colony_id,
        'Date_InitialTag': group['Date_InitialTag'].iloc[0],
        'Month_year': month,
            
        # col containing results of the function that matches statuses
        'Match': status_match(group),
            
        'colony_condition': group['colony_condition'].iloc[0],
        'sample_statuses': group['Health_status'].tolist(),
        'sample_ids': group['Tubelabel_species'].tolist(),
        'mortality_date' : group['Date_DocumentedMortality'].iloc[0]
})
past_matches = pd.DataFrame(rows)

In [376]:
# where are there mismatches? 
past_matches[past_matches['Match']==False]

# just missing one disease margin sample 

,colony_id,Month_year,Match,colony_condition,sample_statuses,Date_InitialTag
3,T1_19_PAST,52022,False,Diseased,[Diseased_Tissue],6/21/19


In [383]:
# create for loop for each specie (of my samples) 
species_list=matched_metadata['Species'].unique().tolist()
species_list

species_checks = {}

for specie in species_list: 
    # filter colony and sample data for each specie 
    filtered_colony=colony[colony['Species']==specie]

    # reduce cols and pivot 
    conditions_long = (
        filtered_colony.loc[:, ['colony_id', '062019_Condition', '052022_Condition', '122022_Condition']]
                   .melt(id_vars='colony_id', var_name='Month_year', value_name='colony_condition')
    )
    conditions_long['Month_year'] = conditions_long['Month_year'].str.extract(r'(\d+)').astype(int)
   
    # repeat for sample data 
    filtered_samples=matched_metadata[matched_metadata['Species']==specie]
    filtered_samples=filtered_samples.loc[:,('colony_id','Month_year','Health_status','Sampling_notes','Tubelabel_species')] 
    filtered_samples['sample_condition']=filtered_samples['Health_status']
    
    # merge dfs 
    merged = pd.merge(
    conditions_long.merge(filtered_colony, on='colony_id', how='left'),
    filtered_samples,
    on=['colony_id', 'Month_year'],
    how='outer'
    )
    
    checks = []
        # check each 'group' (unique combos of colony and monthyear) and store results 
    for (colony_id, month), group in merged.groupby(['colony_id', 'Month_year']):
        checks.append({
            'colony_id': colony_id,
            'Date_InitialTag': group['Date_InitialTag'].iloc[0],
            'Month_year': month,
                
            # col containing results of the function that matches statuses
            'Match': status_match(group),
                
            'colony_condition': group['colony_condition'].iloc[0],
            'sample_statuses': group['Health_status'].tolist(),
            'sample_ids': group['Tubelabel_species'].tolist(),
            'mortality_date' : group['Date_DocumentedMortality'].iloc[0]
    })
    # store sp checks in dict
    species_checks[specie]=pd.DataFrame(checks)
    

In [397]:
false_matches = {}
for specie in species_list:
    df = pd.DataFrame(species_checks[specie])
    false = df[(df['Match'] == False) & (~df['colony_condition'].isna())]
    false_matches[specie] = false

# combine
all_false_matches = pd.concat(
    [df.assign(Species=specie) for specie, df in false_matches.items()],
    ignore_index=True
)

all_false_matches

,colony_id,Date_InitialTag,Month_year,Match,colony_condition,sample_statuses,sample_ids,mortality_date,Species
0,T1_23_OANN,5/21/22,52022.0,False,DC,[Healthy],[052022_BEL_CBC_T1_35_OANN],Healthy,OANN
1,T4_99_OANN,12/5/22,122022.0,False,Diseased,[nan],[nan],Diseased,OANN
2,T1_19_PAST,6/21/19,52022.0,False,Diseased,[Diseased_Tissue],[052022_BEL_CBC_T1_52_PAST],4/1/24,PAST
3,T1_8_MCAV,6/24/19,122022.0,False,Diseased,[Diseased_Tissue],[122022_BEL_CBC_T1_144_MCAV],9/25/23,MCAV
4,T2_59_MCAV,6/21/19,52022.0,False,Diseased,[nan],[nan],Diseased,MCAV
5,T3_67_MCAV,12/3/22,122022.0,False,Diseased,[Diseased_Tissue],[122022_BEL_CBC_T3_145_MCAV],9/25/23,MCAV
6,T3_71_MCAV,12/3/22,122022.0,False,DC,[Healthy],[122022_BEL_CBC_T3_155_MCAV],Healthy,MCAV
